# MetAgeFormer — 4. Lightweight Distilled Model

The distilled **blood-token Transformer + DeepGompertz** model maps a small blood
biochemistry panel directly to embeddings and aging outputs — no NMR backbone,
no tokenizer required. Missing values (NaN) are handled natively via mask embeddings.

**Requirements**
- `Model_Weights/Lightweight/` (`model_weights.pth`, `model_conf.json`)
- `Model_Weights/DeepGompertz/model_weights.pth` (teacher — read-only, to recover head config
  and Gompertz baseline parameters)

Input: a matrix of shape `(n_samples, n_features)` in the same feature order as the
training blood panel (`model_conf.json["n_features"]`), z-scored with the same statistics
as the training data (layer `Z-score normalized`).

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "Src"))

import numpy as np
import torch

from common.constants import AGE_COL
from metageformer_torch.checkpoint import load_teacher_gompertz_config
from metageformer_torch.models import MetAgeFormer_Lightweight_DeepGompertz

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Load the distilled model

`model_conf.json` describes the student architecture; the teacher DeepGompertz checkpoint
supplies `gompertz_head_config` and `baseline_params` (via
`metageformer_torch.checkpoint.load_teacher_gompertz_config`).

In [ ]:
LIGHTWEIGHT_DIR = REPO_ROOT / "Model_Weights" / "Lightweight"
TEACHER_HEAD_PATH = REPO_ROOT / "Model_Weights" / "DeepGompertz" / "model_weights.pth"

for path in (LIGHTWEIGHT_DIR / "model_weights.pth", LIGHTWEIGHT_DIR / "model_conf.json", TEACHER_HEAD_PATH):
    assert path.exists(), f"Missing {path} — download released weights first (Model_Weights/readme.md)"

with open(LIGHTWEIGHT_DIR / "model_conf.json") as f:
    model_conf = json.load(f)

teacher_meta = load_teacher_gompertz_config(str(TEACHER_HEAD_PATH))

model = MetAgeFormer_Lightweight_DeepGompertz(
    model_conf,
    gompertz_head_config=teacher_meta["gompertz_head_config"],
    baseline_params=teacher_meta["baseline_params"],
)
model.from_distilled(str(LIGHTWEIGHT_DIR / "model_weights.pth"))
model.to(DEVICE).eval()

print("student n_features:", model_conf["n_features"])
print("student architecture:", {k: v for k, v in model_conf.items() if k != "n_features"})

## 2. Inference on a synthetic panel (NaN-safe)

For a quick API test, build a random panel with ~10% missing values.

In [ ]:
rng = np.random.default_rng(0)
n_features = int(model_conf["n_features"])
n_samples = 8

demo_x = torch.tensor(rng.normal(0.0, 1.0, size=(n_samples, n_features)), dtype=torch.float32)
demo_x[rng.random(demo_x.shape) < 0.1] = float("nan")  # ~10% missing biomarkers
demo_age = torch.tensor([45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0])

with torch.inference_mode():
    outputs = model(demo_x.to(DEVICE), demo_age.to(DEVICE))

for key, value in outputs.items():
    print(f"{key}: {tuple(value.shape)}")

## 3. Results table

In [ ]:
import pandas as pd

pd.DataFrame({
    "chronological_age": demo_age.numpy(),
    "metabolomic_age": outputs["metabolomic_age"].reshape(-1).cpu().numpy(),
    "age_gap": outputs["age_gap"].reshape(-1).cpu().numpy(),
    "mortality_risk_10y (auxiliary)": outputs["mortality_risk_10y"].reshape(-1).cpu().numpy(),
})

## 4. Inference on a real blood panel (if available)

The training data layout is the same AnnData convention
(layer `Z-score normalized`, obs age column). Feature order must match the released panel.

In [ ]:
BLOOD_PATH = REPO_ROOT / "Data" / "Blood_dataset_fullcohort_107nonderived_mlm" / "val.h5ad"

if BLOOD_PATH.exists():
    import anndata as ad

    blood = ad.read_h5ad(BLOOD_PATH)
    x = torch.tensor(
        np.asarray(blood.layers["Z-score normalized"], dtype=np.float32)[:64],
        device=DEVICE,
    )
    age = torch.tensor(
        blood.obs[AGE_COL].to_numpy()[:64], dtype=torch.float32, device=DEVICE
    )
    assert x.shape[1] == n_features, (
        f"panel width {x.shape[1]} != model n_features {n_features} — "
        "feature order must match the released blood panel"
    )

    with torch.inference_mode():
        out = model(x, age)
    print("blood panel inference OK; embeddings:", tuple(out["embs"].shape))
else:
    print("Blood dataset not found — the synthetic-panel demo above is sufficient for API testing.")